## Microsoft Entra ID 개요

Microsoft Entra ID(이전 Azure Active Directory)는 Microsoft의 클라우드 기반 ID 및 액세스 관리 서비스입니다. Microsoft 365, Azure 및 수천 개의 기타 SaaS 애플리케이션을 위한 중앙 IdP 역할을 합니다.

주요 기능:
* **Single Sign-On(SSO)**: 한 번 인증하여 여러 애플리케이션에 액세스
* **Multi-Factor Authentication(MFA)**: 추가 검증 방법을 통한 보안 강화
* **Conditional Access**: 사용자, 디바이스, 위치, 위험을 기반으로 하는 정책 기반 액세스 제어
* **애플리케이션 통합**: OAuth 2.0, OpenID Connect, SAML 같은 최신 인증 프로토콜 지원

## 학습 목표

Microsoft Entra ID를 AgentCore Identity의 IdP로 사용하여 사용자를 인증하고, 에이전트가 사용자를 대신해 보호된 리소스에 액세스하도록 권한을 부여할 수 있습니다. 이 Notebook에서는 Entra ID를 사용한 Inbound Auth를 살펴봅니다. 

- 사용자가 에이전트를 호출하기 전에 인증합니다.

## Authorization Code Flow
OAuth 2.0 authorization code flow는 웹 애플리케이션에서 사용자를 안전하게 인증하고 access token을 얻기 위한 권장 방식입니다. 

이 흐름은 다음 단계로 구성됩니다.

1. 인증을 위해 사용자를 Entra ID로 리디렉션
2. 로그인 성공 후 authorization code 수신
3. code를 access token 및 refresh token으로 교환
4. 토큰을 사용해 보호된 리소스에 액세스

이 통합 pattern을 사용하면 애플리케이션에 안전한 표준 기반 인증을 유지하면서 AgentCore에서 Entra ID의 강력한 ID 관리 기능을 활용할 수 있습니다.

## 학습 목표 1: AgentCore Identity에서 사용할 Entra ID 설정

### 1단계: Entra ID Tenant 설정

Entra ID tenant는 조직을 나타내는 전용 Microsoft Entra ID 인스턴스입니다. Microsoft 클라우드에 격리된 조직 디렉터리라고 생각할 수 있습니다.

주요 특성:

* **고유한 ID**: 각 tenant에는 고유한 domain이 있습니다(예: yourcompany.onmicrosoft.com).
* **격리된 경계**: 한 tenant의 사용자, 그룹, 애플리케이션은 다른 tenant와 분리됩니다.
* **관리 제어**: tenant 관리자가 사용자, 보안 정책, app registration을 관리합니다.
* **다중 Domain 지원**: 기본 .onmicrosoft.com domain과 함께 custom domain을 포함할 수 있습니다.

실제 적용:

OAuth 2.0 통합을 위해 Entra ID에 애플리케이션을 등록하면 특정 tenant 내부에 등록됩니다. 이후 해당 tenant의 사용자는 조직 자격 증명으로 애플리케이션에 인증할 수 있습니다.

AgentCore 통합에는 다음 항목이 필요합니다.

* **Tenant ID**: Entra ID 인스턴스의 고유 식별자
* **Application Registration**: tenant에 등록된 앱
* **적절한 권한**: 애플리케이션에 구성된 액세스 권한

이 tenant 기반 모델은 인증과 권한 부여가 조직의 보안 경계 안에서 유지되도록 합니다.

tenant 생성 단계는 https://learn.microsoft.com/en-us/entra/fundamentals/create-new-tenant 에서 확인할 수 있습니다.

참고:
1. Microsoft Entra ID는 AWS 서비스가 아닙니다. 비용 관련 정보는 Microsoft Entra ID 문서를 참조하세요.
2. 다음 단계의 화면은 변경될 수 있습니다. Entra ID 애플리케이션 설정에 대한 최신 지침은 Microsoft Entra ID 문서를 참조하세요.

### 2단계: 애플리케이션 설정

1. https://portal.azure.com 으로 이동하여 화면 상단의 검색창에서 "Entra ID"를 검색합니다.
<img src="images/entraid.jpg" width="75%">

2. `Manage` &rarr; `App Registrations`로 이동합니다.
<img src="images/app.registration.png" width="75%">

3. `New Registration`을 클릭하고 세부 정보를 입력합니다. multi-tenant 옵션을 선택하세요.
<img src="images/app.registration.form.png" width="75%">

4. client secret을 생성합니다. AgentCore Identity에서 사용할 clientId와 client secret을 복사합니다.
<img src="images/gather.client.info.png" width="75%">

5. OAuth scope를 생성합니다. Expose an API &rarr; `Add Scope`로 이동하여 전체 scope를 복사해 저장합니다. 
<img src="images/expose.api.png" width="75%">

## 학습 목표 2 - Entra ID Inbound Auth를 사용하는 간단한 에이전트 설정

#### 사전 요구 사항

* Python 3.10+
* AWS 자격 증명
* Amazon Bedrock AgentCore SDK
* Strands Agents
* AWS 리전을 "us-west-2" 또는 Bedrock AgentCore를 지원하는 리전으로 설정. 지원 리전은 https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-regions.html 참조
* 설치된 Docker, Finch 또는 Podman

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet # console/Notebook 출력을 줄이기 위한 quiet mode

In [ ]:
import os
import uuid
import boto3
from boto3.session import Session
from bedrock_agentcore_starter_toolkit import Runtime

boto_session = Session()
sts = boto3.client("sts")
account_id = sts.get_caller_identity().get("Account")
region = boto_session.region_name or "us-west-2"

print(f"AWS Region: {region}")
print(f"AWS Account: {account_id}")

#### 이 Notebook 전반에서 사용할 주요 정보의 환경 변수 설정
audience는 위 2.5단계의 "Application ID URI"와 동일합니다.

다음 값을 확인하세요.
- Tenant ID: "App registration" --> "All Applications" --> 방금 생성한 client 선택 --> "Overview" --> "Directory (tenant) ID"
- Client ID: "App registration" --> "All Applications" --> 방금 생성한 client 선택 --> "Overview" --> "Application (client) ID"
- 앞 단계에서 저장한 secret
- Scope: "App registration" --> "All Applications" --> 방금 생성한 client 선택 --> "Expose a API"의 "Application ID URI"에 "/.default" 접미사 추가
- Audience: "App registration" --> "All Applications" --> 방금 생성한 client 선택 --> "Expose a API"의 "Application ID URI"

In [ ]:
# client_id로 교체
os.environ["client_id"] = "REPLACE_ME"

# secret으로 교체
os.environ["secret"] = "REPLACE_ME"

# scope로 교체
os.environ["scopes"] = "REPLACE_ME"

# tenant_id로 교체
os.environ["tenant_id"] = "REPLACE_ME"

# audience로 교체
os.environ["audience"] = "REPLACE_ME"

#### 에이전트 코드
이 Notebook의 핵심 학습 목표는 Entra ID를 사용하는 Inbound Auth이므로 에이전트는 간단하게 구성합니다.

In [ ]:
%%writefile strands_wo_memory.py
import asyncio

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent

app = BedrockAgentCoreApp()
agent = Agent()

class StreamingQueue:
    def __init__(self):
        self.finished = False
        self.queue = asyncio.Queue()
        
    async def put(self, item):
        await self.queue.put(item)

    async def finish(self):
        self.finished = True
        await self.queue.put(None)

    async def stream(self):
        while True:
            item = await self.queue.get()
            if item is None and self.finished:
                break
            yield item

queue = StreamingQueue()

async def agent_task(user_message: str):
    try:
        await queue.put("Agent execution begins....")
        
        response = agent(user_message)
        # ... 여기에서 에이전트 응답 처리 ...
        await queue.put(response.message)
    except Exception as e:
        await queue.put(f"Failed with error: {repr(e)}")
    finally:
        await queue.put("Agent excecution finished")
        await queue.finish()

@app.entrypoint
async def strands_agent_bedrock(payload, context):
    print("Context object is ....", context)
    prompt = payload.get("prompt", "hello")
    
    task = asyncio.create_task(agent_task(prompt))
    
    async def stream_with_task():
        async for item in queue.stream():
            yield item
        await task
    
    return stream_with_task()

if __name__ == "__main__":
    app.run()


#### Inbound Auth를 적용하도록 `authorizer_configuration`으로 Runtime 구성
Entra ID 기반 Inbound Auth에 `customJWTAuthorizer`를 사용합니다. Tenant ID를 사용해 `discovery_url`을 구성하는 방식을 확인하세요.

In [ ]:
agentcore_runtime = Runtime()

discovery_url = f"https://login.microsoftonline.com/{os.environ['tenant_id']}/.well-known/openid-configuration"

response = agentcore_runtime.configure(
    entrypoint="strands_wo_memory.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_wo_memory_entra_inbound",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedAudience": os.environ["audience"].split(" "),
            # 추가 JWT 권한 부여를 구성할 수 있음. 참고:
            # https://docs.aws.amazon.com/bedrock-agentcore-control/latest/APIReference/API_CustomJWTAuthorizerConfiguration.html
        }
    },
)

print(f"Runtime Agent: {response}")

## 학습 목표 3 - 에이전트를 배포하고 앞에서 받은 bearer token으로 호출

#### Runtime 에이전트 배포
`local_build`가 활성화되어 있으므로 로컬 Docker가 실행 중이어야 합니다. 또는 `local_build=False`로 설정하여 CloudBuild를 사용할 수 있습니다.

In [ ]:
strands_wo_memory_launch_response = agentcore_runtime.launch(
    local_build=False,
    auto_update_on_conflict=True,
)

#### MSAL SDK를 사용해 Entra ID에서 authorization code 가져오기

In [ ]:
import msal

REDIRECT_URI = f"https://bedrock-agentcore.{region}.amazonaws.com/identities/oauth2/callback"
AUTHORITY = f"https://login.microsoftonline.com/{os.environ['tenant_id']}"

app = msal.ConfidentialClientApplication(
    os.environ["client_id"],
    authority=AUTHORITY,
    client_credential=os.environ["secret"],
)

result = app.acquire_token_for_client(scopes=[os.environ["scopes"]])  # scope에는 문자열이 아닌 목록이 필요함
bearer_token_entra = result["access_token"]

In [ ]:
import urllib.parse
import requests
import json

if not strands_wo_memory_launch_response.agent_arn:
    raise Exception(
        "Missing Runtime Agent ARN. Verify that the Runtime Agent was created successfully in the previous step."
    )

escaped_agent_arn = urllib.parse.quote(strands_wo_memory_launch_response.agent_arn, safe="")
url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations?qualifier=DEFAULT"

session_id = str(uuid.uuid1())
headers = {
    "Authorization": f"Bearer {bearer_token_entra}",
    "Content-Type": "application/json",
    "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
    "X-Amzn-Trace-Id": f"entra_id_inbound_sample_{session_id}",
}

http_response = requests.post(
    url,
    data=json.dumps({"prompt": "Hello! I am John Doe. I like brick oven pizza!", "user_id": "user1"}),
    headers=headers,
)
http_response.raise_for_status()


print(f"Agent Response: {http_response.text}")

#### 이 session의 이전 상호 작용은 `agents.messages`를 통해 확인할 수 있습니다. AgentCore Memory는 사용하지 않으므로 새 session ID를 사용하면 에이전트가 이전 상호 작용을 기억하지 못합니다.

In [ ]:
http_response = requests.post(url, data=json.dumps({"prompt": "Who am I?", "user_id": "user1"}), headers=headers)
http_response.raise_for_status()

print(f"Agent Response: {http_response.text}")

#### 또는 AgentCore Runtime 객체로 에이전트를 호출할 수 있습니다. 이전 session을 이어 가려면 bearer token과 동일한 session ID를 전달하세요.

In [ ]:
invoke_response = agentcore_runtime.invoke(
    {"prompt": "Who am I?", "user_id": "user1"},
    bearer_token=bearer_token_entra,
    session_id=session_id,
)

print(f"Agent Invoke Response: {invoke_response}")

## 마무리 및 정리

이 Notebook에서는 다음 내용을 알아보았습니다.

- OAuth 2.0 Authorization Code flow를 제공하도록 Entra ID API와 애플리케이션 설정
- AgentCore Runtime을 생성하고 Entra ID Inbound Auth를 사용하는 에이전트 배포
- 보호된 에이전트에 액세스하기 위한 토큰 획득

#### 생성된 리소스

In [ ]:
print(f"Runtime Agent Arn: {strands_wo_memory_launch_response.agent_id}")

#### AgentCore Runtime 에이전트 삭제

In [ ]:
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)

agentcore_control_client.delete_agent_runtime(agentRuntimeId=strands_wo_memory_launch_response.agent_id)